# ACAF restricted to HapMap3, everyone, once per CDR (read by hm3_panel and grm)

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = min(cfg.params.get("threads", 4), os.cpu_count() or 1)
if cfg.params.get("scratch"):   # mktemp in %%bash, tempfile, and R use TMPDIR
    import tempfile
    tempfile.tempdir = os.environ["TMPDIR"] = os.path.expanduser(cfg.params["scratch"])
    os.makedirs(tempfile.tempdir, exist_ok=True)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

## Which chromosomes are missing

In [ ]:
STAGE = "acaf_hm3"
require_ready(cfg, STAGE)
rep = Report(cfg, STAGE)
items = target_items(cfg, STAGE)
OUTDIR = os.path.dirname(items[0][0])
todo = [it["chr"] for u, it in items if FORCE or not get_store(u).exists(u)]
BACKEND = cfg.params.get("backend", "local")
print(BACKEND, "|", len(todo), "of", len(items), "chromosomes to do ->", OUTDIR)

## One chromosome (runs locally or as the dsub --command)

In [ ]:
# HM3 positions, multi-allelic rows split, IDs chr:pos:ref:alt, biallelic ACGT SNPs. Everyone, no sample filter.
TASK = r"""
set -euo pipefail
mkdir -p "${WORKDIR:-/tmp}" "$OUTDIR"
work=$(mktemp -d -p "${WORKDIR:-/tmp}")
src=${ACAF//\{chr\}/$CHR}
if [[ $src == gs://* ]]; then
  gcloud storage cp ${BILLING:+--billing-project=$BILLING} "$src.pgen" "$src.pvar" "$src.psam" "$work/"
  src="$work/$(basename "$src")"
fi
chmod +x "$PLINK2" 2>/dev/null || true
awk -F: -v OFS='\t' -v c="$CHR" '$1 == c {print $1, $2, $2}' "$HM3_IDS" | sort -k2,2n -u > "$work/pos.bed"
"$PLINK2" --pfile "$src" --extract bed1 "$work/pos.bed" --threads "$THREADS" --silent \
  --make-pgen multiallelics=- --out "$work/split"
"$PLINK2" --pfile "$work/split" --set-all-var-ids '@:#:$r:$a' --new-id-max-allele-len 1000 --extract "$HM3_IDS" \
  --snps-only just-acgt --max-alleles 2 --rm-dup exclude-all --threads "$THREADS" --silent \
  --make-pgen --out "$OUTDIR/hm3.chr$CHR"
rm -rf "$work"
echo "chr$CHR $(grep -vc '^#' "$OUTDIR/hm3.chr$CHR.pvar") variants $(date +%T)"
"""

## Local backend (simulated data, or a VM run without Batch)

In [ ]:
if BACKEND == "local" and todo:
    os.makedirs(OUTDIR, exist_ok=True)
    for c in todo:
        env = {**os.environ, "CHR": str(c), "ACAF": cfg.input("acaf_pfile", chr="{chr}"), "HM3_IDS": cfg.input("hm3_ids"),
               "PLINK2": cfg.tool("plink2"), "OUTDIR": OUTDIR, "THREADS": str(THREADS), "WORKDIR": tempfile.gettempdir()}
        subprocess.run(["bash", "-c", TASK], env=env, check=True)

## dsub backend: one Batch task per chromosome (settings from the 1kg_eur GRM run)

In [ ]:
if BACKEND == "dsub" and todo:
    D, BILL = cfg.params["dsub"], cfg.param("billing_project")
    run = lambda *a: subprocess.run(list(a), check=True, capture_output=True, text=True).stdout
    SA = D.get("service_account") or run("curl", "-s", "-H", "Metadata-Flavor: Google",
        "http://metadata.google.internal/computeMetadata/v1/instance/service-accounts/default/email").strip()
    acaf_gs = cfg.input("acaf_pfile_gs", chr="{chr}")
    ids_gs = f"{OUTDIR}/inputs/hm3_ids_matched.txt"
    run("gcloud", "storage", "cp", cfg.input("hm3_ids"), ids_gs)
    if not get_store(D["plink2"]).exists(D["plink2"]):
        run("gcloud", "storage", "cp", cfg.tool("plink2"), D["plink2"])
    # disk from the real file sizes: the largest chromosome's ACAF files, twice, plus headroom
    size = {c: sum(int(l.split()[0]) for l in run("gcloud", "storage", "du", f"--billing-project={BILL}",
                                                  acaf_gs.replace("{chr}", str(c)) + ".*").splitlines() if l.strip())
            for c in todo}
    disk = int(2 * max(size.values()) / 1e9) + 30
    tasks = os.path.join(tempfile.gettempdir(), "acaf_hm3_tasks.tsv")
    with open(tasks, "w") as fh:
        fh.write("--env CHR\t--output-recursive OUTDIR\n" + "".join(f"{c}\t{OUTDIR}\n" for c in todo))
    print(f"{len(todo)} tasks, {D['machine_type']}, disk {disk} GB, largest ACAF chr {max(size.values())/1e9:.1f} GB, SA {SA}")
    setenv(PROJECT=D["project"], REGION=D["region"], SA=SA, TAG=D["cloud_sdk_tag"], LOGGING=D["logging"],
           MACHINE=D["machine_type"], DISK=disk, TASKS=tasks, IDS_GS=ids_gs, PLINK_GS=D["plink2"],
           ACAF_GS=acaf_gs, BILL=BILL, TTHREADS=D.get("threads", 8), TASK=TASK)

In [ ]:
%%bash
if [ -z "${TASKS:-}" ] || [ ! -s "${TASKS:-/nonexistent}" ]; then echo "nothing to submit"; exit 0; fi
set -euo pipefail
# dsub's wrapper image tag is withdrawn from gcr.io in every release: patch in the same shell as the submit
DSUB_DIR=$(python -c "import dsub, os; print(os.path.dirname(dsub.__file__))")
sed -i -E "s#cloud-sdk:([0-9]+\.[0-9]+\.[0-9]+-)?slim#cloud-sdk:$TAG#g" "$DSUB_DIR/providers/google_utils.py"
find "$DSUB_DIR" -name '*.pyc' -delete
grep -q "cloud-sdk:$TAG" "$DSUB_DIR/providers/google_utils.py" || { echo "wrapper image patch did not apply"; exit 1; }
dsub --provider google-batch --project "$PROJECT" --regions "$REGION" --logging "$LOGGING" \
  --service-account "$SA" --network "projects/$PROJECT/global/networks/network" \
  --subnetwork "projects/$PROJECT/regions/$REGION/subnetworks/subnetwork" --use-private-address \
  --image "gcr.io/google.com/cloudsdktool/cloud-sdk:$TAG" --name acaf-hm3 \
  --machine-type "$MACHINE" --disk-size "$DISK" \
  --input HM3_IDS="$IDS_GS" --input PLINK2="$PLINK_GS" \
  --env ACAF="$ACAF_GS" --env BILLING="$BILL" --env THREADS="$TTHREADS" --env WORKDIR=/mnt/data/work \
  --tasks "$TASKS" --command "$TASK" --wait

## Check and record

In [ ]:
def text(uri):
    return get_store(uri).read_text(uri)

missing = [u for u, _ in items if not get_store(u).exists(u)]
rep.check("all 22 chromosomes written", not missing, ", ".join(os.path.basename(u) for u in missing))
n_var = {it["chr"]: sum(1 for l in text(u[:-5] + ".pvar").splitlines() if l and not l.startswith("#"))
         for u, it in items if u not in missing}
n_ids = sum(1 for _ in open(cfg.input("hm3_ids")))
rep.table(pd.DataFrame({"chr": list(n_var), "variants": list(n_var.values())}), "variants_by_chr")
rep.note(n_variants=sum(n_var.values()), n_hm3_ids=n_ids)
rep.check("nearly every matched HM3 SNP is in the extract", sum(n_var.values()) >= 0.98 * n_ids,
          f"{sum(n_var.values()):,} of {n_ids:,}")
if not missing:
    rep.count("n_samples", sum(1 for l in text(items[0][0][:-5] + ".psam").splitlines() if l and not l.startswith("#")))
for u, it in items:
    if u not in missing and not get_store(u + ".meta.json").exists(u + ".meta.json"):
        mark_done(STAGE, u, item=it, backend=BACKEND)
rep.save()